# Embedding report - bge-m3 over the OSS corpus

Reads the vectors produced by `ingest/embed_bge_m3.py` and explains them: what text
went in, how it was truncated, what the vector space looks like, where the near
duplicates are, and whether sample queries retrieve sensible works (including
Indonesian queries against English abstracts, which is why bge-m3 was chosen).

Run the embedder first:

```sh
uv run python ingest/embed_bge_m3.py            # full corpus, resumable
uv run python ingest/embed_bge_m3.py --limit 300   # smoke test
```

Outputs: CSVs to `data/embeddings/` (gitignored) and PNG charts to
`report/figures/` (committed). Run with `jupyter nbconvert --execute eval/embed.ipynb`.


In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.decomposition import PCA
from transformers import AutoTokenizer

REPO = Path.cwd()
if not (REPO / "data").exists() and (REPO.parent / "data").exists():
    REPO = REPO.parent
sys.path.insert(0, str(REPO))
EMB_DIR = REPO / "data" / "embeddings" / "bge-m3"
FIG_DIR = REPO / "report" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

if not (EMB_DIR / "embeddings.npy").exists():
    raise SystemExit("no vectors yet - run: uv run python ingest/embed_bge_m3.py")

print("torch", torch.__version__, "| cuda available:", torch.cuda.is_available())
print("vectors:", EMB_DIR / "embeddings.npy")

## What was embedded

`meta.json` records the exact configuration, so the notebook never has to guess
which model produced these vectors.

In [ ]:
meta = json.loads((EMB_DIR / "meta.json").read_text())
embeddings = np.load(EMB_DIR / "embeddings.npy")
ids = [json.loads(line) for line in open(EMB_DIR / "work_ids.jsonl", encoding="utf-8") if line.strip()]
meta["works_on_disk"] = len(embeddings)
meta["ids_on_disk"] = len(ids)
print(json.dumps(meta, indent=2))
assert embeddings.shape[0] == len(ids) == meta["works"], "vectors and ids must line up"
assert embeddings.shape[1] == meta["dim"], "dimension disagrees with meta.json"
assert np.isfinite(embeddings).all(), "non-finite values in the vectors"
print("\nshape:", embeddings.shape, embeddings.dtype, "| bytes:", embeddings.nbytes)

## Corpus metadata aligned to the vectors

Row *i* of the matrix belongs to line *i* of the corpus, so the tables below can
join vectors with titles, years and topics. This alignment is asserted rather
than assumed.

In [ ]:
from ingest.embed_bge_m3 import work_text

records = []
with open(REPO / "data" / "raw" / "openalex_works.jsonl", encoding="utf-8") as fh:
    for line in fh:
        w = json.loads(line)
        records.append({
            "id": w["id"], "title": w.get("title"), "year": w.get("publication_year"),
            "language": w.get("language"),
            "topic": (w.get("primary_topic") or {}).get("display_name"),
            "cited_by": w.get("cited_by_count"),
            "abstract_words": len((w.get("abstract") or "").split()),
            "text": work_text(w),
        })
df = pd.DataFrame(records).iloc[: len(embeddings)].reset_index(drop=True)
assert list(df["id"]) == ids, "vector order does not match the corpus order"
print("aligned works:", len(df), "| years", df["year"].min(), "-", df["year"].max())
df[["year", "abstract_words", "cited_by"]].describe().round(1)

## Token length and truncation

`max_seq_length` in `meta.json` is the hard cut-off; anything longer is silently
truncated to its first N tokens. Measured with the same tokenizer the model uses,
this says how much of the corpus actually reaches the vector, and how many works
lose their ending.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(meta["model"])
token_lengths = np.array([
    len(tokenizer(t, add_special_tokens=True)["input_ids"]) for t in df["text"].tolist()
])
max_seq = meta["max_seq"]
truncated = int((token_lengths > max_seq).sum())
print(pd.Series(token_lengths).describe().round(1).to_string())
print(f"\nmax_seq={max_seq}: truncated {truncated}/{len(token_lengths)} = {truncated / len(token_lengths):.2%}")
pd.DataFrame({"token_length": token_lengths, "truncated": token_lengths > max_seq}).to_csv(
    EMB_DIR / "token_lengths.csv", index=False)

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(token_lengths, bins=60)
ax.axvline(max_seq, color="crimson", linestyle="--", label=f"max_seq_length={max_seq}")
ax.set_title(f"Token length of title + abstract ({truncated} works truncated)")
ax.set_xlabel("tokens")
ax.set_ylabel("works")
ax.legend()
fig.tight_layout()
fig.savefig(FIG_DIR / "embed_token_length.png", dpi=120)

## Vector norms

bge-m3 returns unnormalised CLS-pooled vectors. The norm distribution is a sanity
check: a healthy run has a tight, bounded spread, while a collapsed or exploded
spread signals fp16 trouble or a truncated batch. It is also the reason the
similarity analysis below normalises before taking dot products, and why the
Qdrant collection should use cosine distance.

In [ ]:
norms = np.linalg.norm(embeddings, axis=1)
print(norms.describe(percentiles=[0.01, 0.10, 0.90, 0.99]).round(3).to_string())
print("\nzero vectors:", int((norms == 0).sum()))
pd.Series({"mean": norms.mean(), "std": norms.std(), "min": norms.min(), "max": norms.max()}).to_csv(
    EMB_DIR / "norm_summary.csv", header=["value"])

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(norms, bins=60)
ax.set_title("L2 norm of the embedding vectors")
ax.set_xlabel("norm")
ax.set_ylabel("works")
fig.tight_layout()
fig.savefig(FIG_DIR / "embed_norms.png", dpi=120)

## How similar are works to each other?

Two views on a random sample: random pairs (the background level of similarity,
which is what unrelated works score) and each sampled work's nearest neighbour
(the top of its ranking). A nearest-neighbour distribution that sits clearly above
the random background means the space discriminates; pairs above ~0.9 are near
duplicates worth looking at, since they usually mean the same work indexed twice
or a replication package next to its paper.

In [ ]:
rng = np.random.default_rng(42)
sample = rng.choice(len(embeddings), size=min(1500, len(embeddings)), replace=False)
unit = embeddings[sample] / norms[sample, None]
similarity = unit @ unit.T
pairs = similarity[np.triu_indices_from(similarity, k=1)]
nearest = np.sort(similarity, axis=1)[:, -2]

print(f"random pairs : mean {pairs.mean():.3f}  p95 {np.quantile(pairs, 0.95):.3f}  p99 {np.quantile(pairs, 0.99):.3f}")
print(f"nearest neighbour: mean {nearest.mean():.3f}  p05 {np.quantile(nearest, 0.05):.3f}  p95 {np.quantile(nearest, 0.95):.3f}")

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(pairs, bins=60, alpha=0.75, label="random pairs")
ax.hist(nearest, bins=60, alpha=0.75, label="nearest neighbour")
ax.set_title("Cosine similarity: background vs nearest neighbour")
ax.set_xlabel("cosine similarity")
ax.set_ylabel("count")
ax.legend()
fig.tight_layout()
fig.savefig(FIG_DIR / "embed_similarity.png", dpi=120)

full = embeddings / norms[:, None]
in_sample = np.argsort(similarity, axis=1)[:, -2]
rows = [{"score": round(float(similarity[i, in_sample[i]]), 4),
         "work_a": df.loc[sample[i], "title"], "year_a": df.loc[sample[i], "year"],
         "work_b": df.loc[sample[in_sample[i]], "title"], "year_b": df.loc[sample[in_sample[i]], "year"]}
        for i in range(len(sample))]
dups = pd.DataFrame(rows).query("score > 0.9").sort_values("score", ascending=False)
dups.to_csv(EMB_DIR / "near_duplicate_pairs.csv", index=False)
print(f"\npairs above 0.90 of {len(rows)} sampled: {len(dups)}")
print(dups.head(10).to_string(index=False))

## PCA projection

A linear projection to two dimensions compresses 1024 dims, so read it as a rough
sketch, not a map: some structure survives, some does not. Colouring by year asks
whether the corpus drifts over time; colouring by primary topic asks whether the
topic labels agree with the geometry the embedding found on its own.

In [ ]:
proj_sample = rng.choice(len(embeddings), size=min(3000, len(embeddings)), replace=False)
pca = PCA(n_components=2, random_state=42)
coords = pca.fit_transform(embeddings[proj_sample])
print("explained variance:", pca.explained_variance_ratio_.round(3), "total",
      round(pca.explained_variance_ratio_.sum(), 3))
info = df.iloc[proj_sample].reset_index(drop=True)

fig, ax = plt.subplots(figsize=(8, 6))
scatter = ax.scatter(coords[:, 0], coords[:, 1], c=info["year"], cmap="viridis", s=6, alpha=0.7)
fig.colorbar(scatter, ax=ax, label="publication year")
ax.set_title("PCA of bge-m3 vectors, coloured by year")
fig.tight_layout()
fig.savefig(FIG_DIR / "embed_pca_year.png", dpi=120)

top_topics = info["topic"].value_counts().head(8).index
groups = info["topic"].where(info["topic"].isin(top_topics), "other")
fig, ax = plt.subplots(figsize=(9, 6))
for label in list(top_topics) + ["other"]:
    mask = (groups == label).to_numpy()
    ax.scatter(coords[mask, 0], coords[mask, 1], s=6, alpha=0.7, label=f"{label[:28]} ({mask.sum()})")
ax.legend(fontsize=7, loc="best")
ax.set_title("PCA of bge-m3 vectors, coloured by primary topic")
fig.tight_layout()
fig.savefig(FIG_DIR / "embed_pca_topic.png", dpi=120)

## Sanity queries

Five English and three Indonesian queries, drawn from the stakeholder question
list. This is the qualitative half of the checklist's exit criterion ("sample
queries return sensible results"): the retrieval numbers live in the later
Recall@K experiments, but if the top hits here are unrelated, something is wrong
with the vectors, the text template, or the ordering.

bge-m3 needs no query prefix (unlike e5 models), so queries are encoded exactly
like documents.

In [ ]:
from ingest.embed_bge_m3 import load_model, resolve_device

QUERIES = [
    ("en", "What factors influence the adoption of open source software in public organisations?"),
    ("en", "What are the main barriers when migrating from proprietary software to open source?"),
    ("en", "How does vendor lock-in affect the decision to migrate to open source software?"),
    ("en", "What role do government policies play in promoting open source software adoption?"),
    ("en", "Which studies report migrating office systems to LibreOffice in public institutions?"),
    ("id", "Apa saja penelitian tentang penggunaan open source di pemerintah Indonesia?"),
    ("id", "Faktor apa yang membuat organisasi enggan beralih dari perangkat lunak berbayar?"),
    ("id", "Bagaimana tren penelitian transisi ke open source dari tahun ke tahun?"),
]

device = resolve_device("auto")
model = load_model(meta["model"], device, meta["max_seq"])
print("query encoding on:", device, "| vectors were built on:", meta.get("device"))

query_vectors = model.encode([q for _, q in QUERIES], batch_size=8, convert_to_numpy=True,
                             normalize_embeddings=False).astype(np.float32)
query_unit = query_vectors / np.linalg.norm(query_vectors, axis=1, keepdims=True)
scores = query_unit @ full.T
top = np.argsort(-scores, axis=1)[:, :5]

rows = []
for qi, (lang, question) in enumerate(QUERIES):
    print(f"\n[{lang}] {question}")
    for rank, wi in enumerate(top[qi], start=1):
        rows.append({"query": question, "lang": lang, "rank": rank,
                     "score": round(float(scores[qi, wi]), 4), "year": df.loc[wi, "year"],
                     "topic": df.loc[wi, "topic"], "title": df.loc[wi, "title"]})
        print(f"  {rank}. {scores[qi, wi]:.3f}  ({df.loc[wi, 'year']}) {str(df.loc[wi, 'title'])[:88]}")

hits = pd.DataFrame(rows)
hits.to_csv(EMB_DIR / "query_top5.csv", index=False)

fig, ax = plt.subplots(figsize=(9, 5))
for qi, (lang, question) in enumerate(QUERIES):
    ax.plot(range(1, 6), scores[qi, top[qi]], marker="o",
            linestyle="--" if lang == "id" else "-", label=f"[{lang}] {question[:48]}")
ax.set_title("Top-5 cosine similarity per sample query")
ax.set_xlabel("rank")
ax.set_ylabel("cosine similarity")
ax.legend(fontsize=6, loc="upper right")
fig.tight_layout()
fig.savefig(FIG_DIR / "embed_query_scores.png", dpi=120)

## Summary for BAB 3

The numbers below are the ones worth quoting in the methodology chapter, together
with the truncation and near-duplicate caveats.

In [ ]:
summary = {
    "model": meta["model"], "revision": meta.get("revision"), "dim": int(embeddings.shape[1]),
    "works": int(embeddings.shape[0]), "text_template": meta.get("text_template"),
    "device": meta.get("device"), "batch_used": meta.get("batch_used"),
    "max_seq": meta.get("max_seq"), "embedding_seconds": meta.get("seconds"),
    "works_per_second": round(meta["works"] / meta["seconds"], 2) if meta.get("seconds") else None,
    "tokens_mean": round(float(token_lengths.mean()), 1),
    "tokens_p95": round(float(np.quantile(token_lengths, 0.95)), 1),
    "truncated_works": truncated, "truncated_share": round(truncated / len(token_lengths), 4),
    "norm_mean": round(float(norms.mean()), 3), "norm_std": round(float(norms.std()), 3),
    "random_pair_similarity_mean": round(float(pairs.mean()), 3),
    "nearest_similarity_mean": round(float(nearest.mean()), 3),
    "near_duplicate_pairs_above_0_90": int(len(dups)),
    "query_top1_min": round(float(scores[np.arange(len(QUERIES)), top[:, 0]].min()), 3),
    "query_top1_mean": round(float(scores[np.arange(len(QUERIES)), top[:, 0]].mean()), 3),
}
(EMB_DIR / "summary.json").write_text(json.dumps(summary, indent=2) + "\n")
print(json.dumps(summary, indent=2))